# Optimizaition

### 1.Draw a simple function

In [ ]:
import torch
import matplotlib.pyplot as plt

# Input range
u = torch.linspace(-3.0, 3.0, 2000)

# Function
fu = (u * torch.sin(u) - 1.0) ** 2

# Plot
plt.figure(figsize=(9, 5))
plt.plot(u.numpy(), fu.numpy(), label=r"$f(u)=(u\sin(u)-1)^2$")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("u")
plt.ylabel("f(u)")
plt.title("Function Plot")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### 2. Automatic Differentiation

In [ ]:
import torch
u = torch.tensor([0.01], # why cannot be [0.0]
                dtype=torch.float64,
                requires_grad=True,
                device='cpu')
fu = (u * torch.sin(u) - 1)**2
fu.backward()
print(u.grad)

### 3. Gradient Descent

In [ ]:
import torch
u = torch.tensor([-0.01], # why cannot be [0.0]
                dtype=torch.float64,
                requires_grad=True,
                device='cpu')
optimizer = torch.optim.SGD([u], lr=0.01)
for it in range(1000):
   optimizer.zero_grad()
   fu = (u * torch.sin(u) - 1)**2
   fu.backward()
   optimizer.step()
print(u)

# Inverse Kinematics

### 0. Load Curobo header

In [ ]:
import copy
from curobo.config_io import load_yaml
from curobo.content import get_robot_configs_path
from curobo.viewer import ViserVisualizer 
from curobo.types import GoalToolPose, JointState, Pose
import torch
from IPython.display import IFrame, display

robot_config = load_yaml(str(get_robot_configs_path() / "franka.yml"))
robot_config["robot_cfg"]["kinematics"]["tool_frames"] = ["ee_link"]

try:
    viz._server.stop()
except NameError:
    pass

viz = ViserVisualizer(content_path=copy.deepcopy(robot_config))

### 1. Gradient descent IK

In [ ]:
def draw_frame(viz, pose, name):
    frame = viz.add_frame(name, pose)
    frame.axes_length = 0.1
    frame.axes_radius = 0.005
    frame.origin_radius = 0.01
F0 = Pose.from_list([0.5, 0.0, 0.1, 0.0, 1.0, 0.0, 0.0])
draw_frame(viz, F0, "/F0")
viz._server.scene.show()

In [ ]:
# compute IK loss
u = viz._kinematics.default_joint_state
Fu = viz._kinematics.compute_kinematics(u).tool_poses.get_link_pose("ee_link")
T_F0 = F0.get_matrix()
T_Fu = Fu.get_matrix()
print(torch.linalg.norm(T_F0 - T_Fu))

In [ ]:
# gradient descent to solve IK
port = viz._server.get_port()
display(
    IFrame(
        src=f"http://localhost:{port}",
        width="100%",
        height=700,
    )
)
q0 = viz._kinematics.default_joint_state.position
q = q0.detach().clone().requires_grad_(True)
optimizer = torch.optim.Adam([q], lr=0.001)
for step in range(10000):
    optimizer.zero_grad()
    joint_state = JointState.from_position(q,joint_names=viz.joint_names)
    kin_state = viz._kinematics.compute_kinematics(joint_state)
    current_pose = kin_state.tool_poses.get_link_pose("ee_link")
    loss = torch.sum((current_pose.position - F0.position)**2)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        draw_frame(viz, current_pose.detach(), "/Fu")
        viz.set_joint_state(joint_state.detach().clone())

### 2. Curobo IK

In [ ]:
from curobo.inverse_kinematics import InverseKinematics, InverseKinematicsCfg
robot_config = load_yaml(str(get_robot_configs_path() / "franka.yml"))
robot_config["robot_cfg"]["kinematics"]["tool_frames"] = ["ee_link"]

try:
    viz._server.stop()
except NameError:
    pass

viz = ViserVisualizer(content_path=copy.deepcopy(robot_config))

#### Solve Single IK

In [ ]:
# Solve Single ik
ik = InverseKinematics(InverseKinematicsCfg.create(robot=robot_config))
goal = GoalToolPose.from_poses({"ee_link": F0}, num_goalset=1)
F0 = Pose.from_list([0.5, 0.0, 0.025, 0.0, 1.0, 0.0, 0.0])
result = ik.solve_pose(goal)
solution = result.js_solution.squeeze(0)

# render
viz.set_joint_state(solution)
active_solution = ik.get_active_js(solution)
Fu = ik.compute_kinematics(active_solution).tool_poses.get_link_pose("ee_link")
draw_frame(viz, Fu, "/Fu")
viz._server.scene.show()

#### Solve Multiple IK

In [ ]:
ik = InverseKinematics(InverseKinematicsCfg.create(robot=robot_config,  num_seeds=64))
goal = GoalToolPose.from_poses({"ee_link": F0}, num_goalset=1)
result = ik.solve_pose(goal, return_seeds = 64)
js_solutions = result.js_solution.squeeze(0)[result.success.squeeze(0)]

# render
slider = viz._server.gui.add_slider("/seed_index", min=0, max=js_solutions.shape[0] - 1, step=1, initial_value=0)
@slider.on_update
def _(_):
    seed_index = int(slider.value)
    solution = js_solutions[seed_index].position
    joint_state = JointState.from_position(solution,joint_names=viz.joint_names)
    viz.set_joint_state(joint_state)

# display
port = viz._server.get_port()
display(
    IFrame(
        src=f"http://localhost:{port}",
        width="100%",
        height=700,
    )
)

#### Grasp Cube

In [ ]:
# add cube
import trimesh
from scipy.spatial.transform import Rotation as R
import numpy as np
box = trimesh.primitives.Box(extents=[0.05, 0.05, 0.05])
box.apply_translation([0.5, 0.0, 0.025])
viz.add_mesh(box, "/box")

rot_matrix = np.array([[0, -1, 0],
                       [0, 0, 1],
                       [-1, 0, 0]])
quat = R.from_matrix(rot_matrix).as_quat(scalar_first = True)
F0 = Pose.from_list([.5, 0, .025, quat[0], quat[1], quat[2], quat[3]])
draw_frame(viz, F0, "/F0")

# solve ik
ik = InverseKinematics(InverseKinematicsCfg.create(robot=robot_config,  num_seeds=1))
goal = GoalToolPose.from_poses({"ee_link": F0}, num_goalset=1)
result = ik.solve_pose(goal, return_seeds = 1)
solution = result.js_solution.squeeze(0)
print("IK solution:", solution.position.detach().cpu().numpy())

# render
viz.set_joint_state(solution)
active_solution = ik.get_active_js(solution)
viz._server.scene.show()

(viser) Connection closed (14, 2 total)

(viser) Connection closed (16, 1 total)

(viser) Connection closed (15, 0 total)

#### Batch IK

In [ ]:
# Rotation group
rots = R.create_group("O")
for i in range(4):
    print(rots[i])

In [ ]:
# batch frames
batch_F0 = []
for i in range(rots.shape[0]):
    quat = rots[i].as_quat(scalar_first = True).tolist()
    batch_F0.append([.5, 0, .025, quat[0], quat[1], quat[2], quat[3]])
print(batch_F0)
batch_F0 = Pose.from_batch_list(batch_F0)
print(batch_F0.position.shape)     # torch.Size([24, 3])
print(batch_F0.quaternion.shape)   # torch.Size([24, 4])

In [ ]:
ik = InverseKinematics(InverseKinematicsCfg.create(
    robot=robot_config,
    num_seeds=1,
    max_batch_size=24))
goal = GoalToolPose.from_poses({"ee_link": batch_F0}, num_goalset=1)
result = ik.solve_pose(goal, return_seeds=1)

# draw IK solutions
js_solutions = result.js_solution.squeeze(0)[result.success.squeeze(0)]
# render
slider = viz._server.gui.add_slider("/seed_index", min=0, max=js_solutions.shape[0] - 1, step=1, initial_value=0)
@slider.on_update
def _(_):
    seed_index = int(slider.value)
    solution = js_solutions[seed_index].position
    joint_state = JointState.from_position(solution,joint_names=viz.joint_names)
    viz.set_joint_state(joint_state)
# display
port = viz._server.get_port()
display(
    IFrame(
        src=f"http://localhost:{port}",
        width="100%",
        height=700,
    )
)

# Collision-Avoidance

#### 0. Load Curobo Head

In [ ]:
import copy
from curobo.config_io import load_yaml
from curobo.content import get_robot_configs_path
from curobo.viewer import ViserVisualizer
from curobo.types import GoalToolPose, JointState, Pose
import torch
from curobo.inverse_kinematics import InverseKinematics, InverseKinematicsCfg
from IPython.display import IFrame, display
from pathlib import Path
import trimesh
from scipy.spatial.transform import Rotation as R
robot_config = load_yaml(str(get_robot_configs_path() / "franka.yml"))
robot_config["robot_cfg"]["kinematics"]["tool_frames"] = ["ee_link"]

try:
    viz._server.stop()
except NameError:
    pass

viz = ViserVisualizer(content_path=copy.deepcopy(robot_config), visualize_robot_spheres = True)
viz._server.scene.show()

#### 1. Load Scene

In [ ]:
# add to scene
ik_cfg = InverseKinematicsCfg.create(
    robot=robot_config,
    scene_model="collision_table.yml",
)
scene = ik_cfg.scene_collision_cfg.scene_model
viz.add_scene(scene)
viz._server.scene.show()

### 2. Load Custom Scene

In [ ]:
from pathlib import Path
ik_cfg = InverseKinematicsCfg.create(
    robot=robot_config,
    scene_model=str(Path.cwd() / "collision_table2.yml")
)
scene = ik_cfg.scene_collision_cfg.scene_model
viz.add_scene(scene)
viz._server.scene.show()

#### 3. Collision-free Batch IK

In [ ]:
# Rotation group
rots = R.create_group("O")
batch_F0 = []
for i in range(rots.shape[0]):
    quat = rots[i].as_quat(scalar_first = True).tolist()
    batch_F0.append([.5, 0, .025, quat[0], quat[1], quat[2], quat[3]])
batch_F0 = Pose.from_batch_list(batch_F0)

ik_cfg = InverseKinematicsCfg.create(
    robot=robot_config,
    scene_model=str(Path.cwd() / "collision_table2.yml"),
    num_seeds=1,
    max_batch_size=24
)
ik = InverseKinematics(ik_cfg)
goal = GoalToolPose.from_poses({"ee_link": batch_F0}, num_goalset=1)
result = ik.solve_pose(goal, return_seeds=1)

# draw IK solutions
js_solutions = result.js_solution.squeeze(0)[result.success.squeeze(0)]
# render
slider = viz._server.gui.add_slider("/seed_index", min=0, max=js_solutions.shape[0] - 1, step=1, initial_value=0)
@slider.on_update
def _(_):
    seed_index = int(slider.value)
    solution = js_solutions[seed_index].position
    joint_state = JointState.from_position(solution,joint_names=viz.joint_names)
    viz.set_joint_state(joint_state)
    
# display
port = viz._server.get_port()
display(
    IFrame(
        src=f"http://localhost:{port}",
        width="100%",
        height=700,
    )
)


#### 4. Collision-free Goalset IK

In [ ]:
ik_cfg = InverseKinematicsCfg.create(
    robot=robot_config,
    scene_model=str(Path.cwd() / "collision_table2.yml"),
    num_seeds=1,
    max_goalset=24
)
ik = InverseKinematics(ik_cfg)
goal = GoalToolPose.from_poses({"ee_link": batch_F0}, num_goalset=batch_F0.position.shape[0])
result = ik.solve_pose(goal)
print(result.js_solution.shape)
viz.set_joint_state(result.js_solution[0, 0])
scene = ik_cfg.scene_collision_cfg.scene_model
viz.add_scene(scene)
viz._server.scene.show()

#### 5. Check Collision

In [ ]:
from curobo.collision_checking import (
    RobotCollisionChecker,
    RobotCollisionCheckerCfg,
)
q = torch.tensor([0.0, 1.57, 0.0, -2.36, 0.0, 1.57, 0.785], device="cuda").reshape(1, 1, 7)
joints = JointState.from_position(q, joint_names=viz.joint_names)
checker_cfg = RobotCollisionCheckerCfg.load_from_config(
    robot_config="franka.yml",
    scene_model=str(Path.cwd() / "collision_table2.yml"),
    collision_activation_distance=0.01,
)
checker = RobotCollisionChecker(checker_cfg)
print(checker.validate(q)) ## False

viz.set_joint_state(joints)
viz._server.scene.show()